# EyeAI Product Backend V1.1.2 — Final Grounded Assistant Integration

This notebook verifies the complete backend before frontend development:

`authentication → readable IDs → patient → visit → RETFound + TTA + heatmap → spatial metrics → optional RAG → grounded Qwen assistant → inline citations → GPU memory`

The notebook performs no training.

## 1. Paths and resource controls

In [ ]:
from pathlib import Path
import importlib
import json
import os
import shutil
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")
API_CONFIG = REPO_DIR / "configs/api/product_backend_v11_assistant.yaml"
SMOKE_ROOT = Path("/kaggle/working/eyeai_product_backend_v112_smoke")

RESET_SMOKE_DATABASE = True
RAG_ENABLED = True
RUN_IMAGE_ANALYSIS = True
RUN_EXPLANATION = True

if RESET_SMOKE_DATABASE and SMOKE_ROOT.exists():
    shutil.rmtree(SMOKE_ROOT)
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
print("Smoke-test workspace:", SMOKE_ROOT)
print("RAG enabled:", RAG_ENABLED)
print("Explanation enabled:", RUN_EXPLANATION)

## 2. Clone and install the final backend

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(REPO_DIR)], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-assistant-kaggle.txt")],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
repo_src = str(REPO_DIR / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
print("Repository and dependencies are ready.")

## 3. Discover model packages, optional RAG, and prepared data

In [ ]:
def resolve_single(candidates, label, required=True):
    candidates = sorted({Path(path).resolve() for path in candidates if Path(path).exists()})
    if len(candidates) == 1:
        return candidates[0]
    if not candidates and not required:
        return None
    raise RuntimeError(f"Expected one {label}, found: {candidates}")

MODEL_PACKAGE_DIR = resolve_single(
    [
        path.parent
        for path in Path("/kaggle/input").rglob("model.pth")
        if (path.parent / "model_config.yaml").is_file()
        and (path.parent / "threshold.json").is_file()
        and (path.parent / "version.json").is_file()
    ],
    "Run 09 TTA Model Package V1",
)
CHAT_MODEL_DIR = resolve_single(
    [
        path.parent
        for path in Path("/kaggle/input").rglob("config.json")
        if "qwen3_4b_instruct_2507" in str(path.parent).lower()
        and (path.parent / "tokenizer_config.json").is_file()
    ],
    "Qwen3 chat model",
)
EMBEDDING_MODEL_DIR = resolve_single(
    [
        path.parent
        for path in Path("/kaggle/input").rglob("config.json")
        if "qwen3_embedding_0_6b" in str(path.parent).lower()
        and (path.parent / "tokenizer_config.json").is_file()
    ],
    "Qwen3 embedding model",
    required=RAG_ENABLED,
)
RAG_INDEX_DIR = resolve_single(
    [
        path.parent
        for path in Path("/kaggle/input").rglob("index.faiss")
        if (path.parent / "chunks.json").is_file()
        and (path.parent / "manifest.json").is_file()
        and (path.parent / "reference_audit.json").is_file()
    ],
    "approved EyeAI RAG index",
    required=RAG_ENABLED,
)
DATASET_ROOT = resolve_single(
    [
        path.parent
        for path in Path("/kaggle/input").rglob("dataset_summary.json")
        if (path.parent / "manifests/hyamd_val.csv").is_file()
    ],
    "prepared EyeAI dataset",
)

print("Model package:", MODEL_PACKAGE_DIR)
print("Chat model:", CHAT_MODEL_DIR)
print("Embedding model:", EMBEDDING_MODEL_DIR)
print("RAG index:", RAG_INDEX_DIR)
print("Prepared dataset:", DATASET_ROOT)

## 4. Configure the isolated backend

In [ ]:
os.environ["EYEAI_DATABASE_URL"] = f"sqlite:///{SMOKE_ROOT / 'eyeai.db'}"
os.environ["EYEAI_EXPLANATION_OUTPUT_DIR"] = str(SMOKE_ROOT / "explanations")
os.environ["EYEAI_REPORTS_OUTPUT_DIR"] = str(SMOKE_ROOT / "reports")
os.environ["EYEAI_JWT_SECRET"] = "kaggle-v112-smoke-secret-change-before-production"
os.environ["EYEAI_ASSISTANT_ENABLED"] = "true"
os.environ["EYEAI_ASSISTANT_MODEL_DIR"] = str(CHAT_MODEL_DIR)
os.environ["EYEAI_RAG_ENABLED"] = "true" if RAG_ENABLED else "false"
if RAG_ENABLED:
    os.environ["EYEAI_RAG_INDEX_DIR"] = str(RAG_INDEX_DIR)
    os.environ["EYEAI_RAG_EMBEDDING_MODEL_DIR"] = str(EMBEDDING_MODEL_DIR)
else:
    os.environ.pop("EYEAI_RAG_INDEX_DIR", None)
    os.environ.pop("EYEAI_RAG_EMBEDDING_MODEL_DIR", None)

from eyeai.api.config import ApiSettings
from eyeai.api.main import create_app

settings = ApiSettings.from_yaml(
    API_CONFIG,
    model_package_override=MODEL_PACKAGE_DIR,
    device_override="cuda" if __import__("torch").cuda.is_available() else "cpu",
    assistant_model_override=CHAT_MODEL_DIR,
    rag_index_override=RAG_INDEX_DIR,
    embedding_model_override=EMBEDDING_MODEL_DIR,
)
print("API version:", settings.version)
print("AMD device:", settings.device)
print("Assistant GPU cap:", settings.assistant_maximum_gpu_memory_gib, "GiB")
print("RAG enabled:", settings.rag_enabled)

## 5. Create the API, administrator, patient, and visit

In [ ]:
from fastapi.testclient import TestClient

app = create_app(settings)
client_context = TestClient(app)
client = client_context.__enter__()

bootstrap = client.post(
    "/api/v1/auth/bootstrap",
    json={
        "email": "doctor.smoke@eyeai.local",
        "full_name": "EyeAI Smoke Doctor",
        "password": "EyeAI-Smoke-Password-2026",
    },
)
bootstrap.raise_for_status()
login = client.post(
    "/api/v1/auth/login",
    json={
        "email": "doctor.smoke@eyeai.local",
        "password": "EyeAI-Smoke-Password-2026",
    },
)
login.raise_for_status()
HEADERS = {"Authorization": f"Bearer {login.json()['access_token']}"}

patient = client.post(
    "/api/v1/patients",
    headers=HEADERS,
    json={
        "medical_record_number": "DEMO-AMD-001",
        "first_name": "Demo",
        "last_name": "Patient",
        "sex": "unspecified",
    },
)
patient.raise_for_status()
PATIENT = patient.json()
visit = client.post(
    f"/api/v1/patients/{PATIENT['display_id']}/visits",
    headers=HEADERS,
    json={"eye": "right", "notes": "Final V1.1.2 integration test."},
)
visit.raise_for_status()
VISIT = visit.json()
print("User:", bootstrap.json()["display_id"])
print("Patient:", PATIENT["display_id"])
print("Visit:", VISIT["display_id"])

## 6. Run RETFound analysis with heatmap spatial metrics

In [ ]:
if RUN_IMAGE_ANALYSIS:
    import pandas as pd

    validation = pd.read_csv(DATASET_ROOT / "manifests/hyamd_val.csv")
    row = validation[validation["binary_label"] == 1].sample(n=1, random_state=42).iloc[0]
    image_path = DATASET_ROOT / row["relative_image_path"]
    with image_path.open("rb") as handle:
        analysis = client.post(
            f"/api/v1/visits/{VISIT['display_id']}/analyze?explanation={'true' if RUN_EXPLANATION else 'false'}",
            headers=HEADERS,
            files={"file": (image_path.name, handle, "image/jpeg")},
        )
    analysis.raise_for_status()
    ANALYSIS = analysis.json()
    print(json.dumps({
        "display_id": ANALYSIS["display_id"],
        "label": ANALYSIS["label"],
        "probability": ANALYSIS["probability"],
        "quality_status": ANALYSIS["quality_status"],
        "explanation_metrics": (ANALYSIS.get("explanation") or {}).get("metrics"),
    }, indent=2, ensure_ascii=False))

    if RUN_EXPLANATION:
        metrics = ANALYSIS["explanation"]["metrics"]
        for key in [
            "peak_x_pixel", "peak_y_pixel", "centroid_x_fraction",
            "centroid_y_fraction", "dominant_region", "tta_map_similarity"
        ]:
            assert key in metrics, f"Missing spatial heatmap metric: {key}"
else:
    print("Image analysis is disabled.")

## 7. Run the grounded, cited clinical assistant

In [ ]:
conversation = client.post(
    f"/api/v1/patients/{PATIENT['display_id']}/assistant/conversations",
    headers=HEADERS,
    json={
        "eye": "right",
        "visit_id": VISIT["display_id"],
        "title": "Right-eye final review",
    },
)
conversation.raise_for_status()
CONVERSATION = conversation.json()

turn = client.post(
    f"/api/v1/assistant/conversations/{CONVERSATION['display_id']}/messages",
    headers=HEADERS,
    json={
        "content": (
            "لخص النتيجة الحالية، وحدد أين ركزت خريطة التأثير بالإحداثيات والوصف المكاني، "
            "واشرح الفائدة السريرية المحدودة بالاستناد إلى المراجع المتاحة مع ذكرها داخل النص."
        )
    },
)
turn.raise_for_status()
RESULT = turn.json()["result"]
print(json.dumps(RESULT, indent=2, ensure_ascii=False))

assert not RESULT["answer"].lstrip().startswith("{")
assert '"answer"' not in RESULT["answer"]
assert RESULT["grounding"]["knowledge_scope"] in {
    "patient_context_only",
    "patient_context_and_approved_rag",
}
if RUN_EXPLANATION:
    assert RESULT["heatmap_spatial"] is not None
    assert RESULT["heatmap_spatial"]["peak"]["pixel"]["x"] >= 0
if RAG_ENABLED:
    assert RESULT["references"], "RAG is enabled but no approved references were returned."
    assert any(f"[{item['citation_number']}]" in RESULT["answer"] for item in RESULT["references"])
    assert RESULT["grounding"]["knowledge_scope"] == "patient_context_and_approved_rag"

for forbidden in [
    "التهاب الشبكية",
    "مرض الشبكية المتقدم",
    "تأكيد التشخيص",
    "التاريخ العائلي",
    "العوامل البيئية",
]:
    assert forbidden not in RESULT["answer"], f"Unsupported phrase detected: {forbidden}"

print("Final assistant contract validated.")

## 8. GPU memory and service health

In [ ]:
import torch

status = client.get("/api/v1/assistant/status", headers=HEADERS)
status.raise_for_status()
print("Assistant status:")
print(json.dumps(status.json(), indent=2))

health = client.get("/health")
health.raise_for_status()
print("AMD service health:")
print(json.dumps(health.json(), indent=2))

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Allocated: {torch.cuda.memory_allocated() / (1024 ** 3):.3f} GiB")
    print(f"Reserved: {torch.cuda.memory_reserved() / (1024 ** 3):.3f} GiB")
    print(f"Peak allocated: {torch.cuda.max_memory_allocated() / (1024 ** 3):.3f} GiB")
    print(f"Peak reserved: {torch.cuda.max_memory_reserved() / (1024 ** 3):.3f} GiB")

client_context.__exit__(None, None, None)
print("Product Backend V1.1.2 is ready for frontend integration.")